# Lab 7 - Sequential workflow を Hosted Agent にする

Microsoft Agent Framework で `policy_agent -> planner_agent -> reviewer_agent` の順に処理する workflow を読み、ローカルで 1 回実行します。

**使用する kernel:** `Python (Foundry Hosted Agent)`

Hosted Agent の remote build は、この Notebook の確認後に `scripts/deploy_hosted_agent.py` で実行します。

## 1. Repository と Foundry project を読み込む

Lab 1 が生成した `.workshop/context.json` から project endpoint と model deployment 名を取得します。`.env` を手で編集する必要はありません。

In [ ]:
import json
import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。Codespace でこの Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
context = json.loads((REPO_ROOT / ".workshop" / "context.json").read_text(encoding="utf-8"))
outputs = context["terraform_outputs"]

os.environ["FOUNDRY_PROJECT_ENDPOINT"] = outputs["foundry_project_endpoint"]["value"]
os.environ["FOUNDRY_MODEL"] = outputs["primary_model_deployment_name"]["value"]

hosted_source = REPO_ROOT / "src" / "hosted-agent"
if str(hosted_source) not in sys.path:
    sys.path.insert(0, str(hosted_source))

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model deployment: {os.environ['FOUNDRY_MODEL']}")

## 2. 3 つの agent の責務を確認する

| Agent | 責務 |
|---|---|
| `policy_agent` | 入力不足と規程上の注意点を整理する |
| `planner_agent` | 規程確認を受けて概算と次の action を作る |
| `reviewer_agent` | 矛盾を直し、最終回答を返す |

3 agent の後に、モデルではない `SimulationNoticeExecutor` が必須の simulation 注意書きを決定的に保証します。instructions と workflow 構築は `src/hosted-agent/workflow.py` が唯一の source of truth です。Notebook にはコピーしません。

In [ ]:
import workflow

print(workflow.WORKFLOW_DESCRIPTION)
print("Sequence: policy_agent -> planner_agent -> reviewer_agent")
print("Final check: SimulationNoticeExecutor")
print("Sample request:")
print(workflow.SAMPLE_REQUEST)

## 3. Workflow を 1 回実行する

`SequentialBuilder` は前の agent の結果を次の agent へ渡します。同じ Foundry model を 3 回呼び出した後、`SimulationNoticeExecutor` が最終出力を検査します。完了まで少し待ちます。

In [ ]:
answer = await workflow.run_workflow(workflow.SAMPLE_REQUEST)
print(answer)

## 4. 最終回答の安全条件を確認する

この sample は予約や承認を行いません。決定的な最終チェックにより、simulation の注意書きが必ず含まれることを確認します。

In [ ]:
assert workflow.SIMULATION_NOTICE in answer
print("OK: 最終回答に simulation の注意書きがあります。")

## 5. Azure を使わない contract test を実行する

fake chat client を使い、agent の順序、message の引き継ぎ、最終回答を短時間で確認します。

In [ ]:
import subprocess

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        str(REPO_ROOT / "tests" / "contract" / "hosted_agent"),
        "-q",
    ],
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    completed.check_returncode()

## 6. Hosted Agent として deploy する

Notebook の確認が終わったら repository root の terminal で次を実行します。

```bash
.venv/bin/python scripts/deploy_hosted_agent.py --output json
```

この script は source を zip 化し、Python 3.13 の remote build を開始し、version が `active` または `failed` になるまで有限時間で待ちます。Docker、ACR、追加の sign-in は不要です。